# Basic RAG — Formulary (Table-Aware)

`01-Basic_Formulary_Tutor.ipynb` was the RAG-less baseline: a plain LLM call, no retrieval, no formulary lookup. This notebook wires retrieval in, following the same **chunk → embed → search → answer** skeleton as `ai-tutor-rag-system/notebooks/02-Basic_RAG.ipynb`, adapted for a PDF/table data source and backporting a few of that notebook's engineering habits that were missing here:

- a **model-keyed embedding cache** (skip re-embedding 2,394 rows on every run)
- **assert vector counts**, not just dimensions, right after embedding
- a **with-context vs. without-context grounding demo**, proving retrieval changes the answer rather than just assuming it does

**What does *not* change:** the chunking strategy. `02-Basic_RAG.ipynb`'s fixed-size character-window chunker is exactly the naive approach `langchain_03_table_aware.ipynb` already diagnosed and replaced for this PDF — splitting a formulary table as raw text put a drug's tier/limits in a different chunk than its name. This notebook reuses that same row-based table extraction (`pdfplumber.extract_tables()`, one `Document` per drug row). Porting `02-Basic_RAG.ipynb`'s chunker here would reintroduce the bug it fixed.

## 1. Setup

In [1]:
# Load configuration from config.yaml — local Ollama LLM + local sentence-transformers
# embeddings, so no API keys are needed anywhere in this notebook.
import yaml
from pathlib import Path

config_path = Path("../../config/config.yaml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

pdf_dir = Path(f"../../{config['paths']['raw_data']}")
pdf_path = next(pdf_dir.glob("*.pdf"))
print(f"✅ Config loaded. PDF: {pdf_path.name}")

✅ Config loaded. PDF: Oscar_4T_NC_STND_Member_Doc__January_2026__as_of_11182025.pdf


## 2. Chunking: Table-Aware, Row-Based (unchanged from `langchain_03_table_aware.ipynb`)

Reusing the extractor that keeps each drug's full record — section, name, tier, requirements — as one self-contained chunk, so a drug's fields can never be split across chunks. This is the PDF-appropriate equivalent of `02-Basic_RAG.ipynb`'s `split_into_chunks()`; the naive fixed-size version does not belong here.

In [2]:
# Extract one Document per drug row, keyed by its section/name/tier/requirements
import pdfplumber
from langchain_core.documents import Document


def extract_drug_rows(path):
    docs = []
    current_section = None  # persists across pages until the next section header

    with pdfplumber.open(path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            for table in page.extract_tables():
                for row in table:
                    cells = [(c or "").strip().replace("\n", " ") for c in row]
                    non_empty = [c for c in cells if c]

                    if not non_empty:
                        continue
                    if "Drug Name" in cells:  # repeated table header row
                        continue
                    if len(non_empty) == 1:  # section header, e.g. "OPIOID ANALGESICS"
                        current_section = non_empty[0]
                        continue
                    if len(cells) < 4:  # page footer / legend row
                        continue

                    _, drug_name, tier, requirements = cells[:4]
                    if not drug_name:
                        continue

                    content = (
                        f"Section: {current_section or 'N/A'}\n"
                        f"Drug: {drug_name}\n"
                        f"Tier: {tier or 'N/A'}\n"
                        f"Requirements/Limits: {requirements or 'None'}"
                    )
                    docs.append(Document(
                        page_content=content,
                        metadata={"section": current_section, "page": page_num, "drug": drug_name},
                    ))
    return docs


row_docs = extract_drug_rows(pdf_path)
assert len(row_docs) > 0, "No drug rows extracted — check the PDF's table layout hasn't changed."
print(f"✅ Extracted {len(row_docs)} drug-row documents")

✅ Extracted 2394 drug-row documents


## 3. Embedding, with a Model-Keyed Cache

**Backported from `02-Basic_RAG.ipynb`.** `langchain_03_table_aware.ipynb` re-embeds all 2,394 rows through the local model on every run — harmless in cost (it's local, not billed per call) but wasteful in time. Same rule as before applies regardless of provider: **a cache belongs to exactly one embedding model.** The cache directory is named after the model ID and dimension, so switching `embedding.model_name` in `config.yaml` simply points at a different (or missing) cache — never a stale one silently reused with the wrong model.

⚠️ `FAISS.load_local` unpickles a docstore file. That's safe here because we only ever load a cache *this notebook itself wrote* — the same principle as `02-Basic_RAG.ipynb` insisting on `json.loads` over `eval()`: never deserialize a file you don't trust the origin of.

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name=config["embedding"]["model_name"])

MODEL_TAG = config["embedding"]["model_name"].replace("/", "-")
CACHE_DIR = Path(f"../../{config['paths']['processed_data']}/faiss_cache_{MODEL_TAG}_{config['embedding']['dimension']}d")
INDEX_NAME = "formulary_rows"

FORCE_REEMBED = False  # set True to rebuild the cache after changing chunking logic or the PDF
cache_exists = (CACHE_DIR / f"{INDEX_NAME}.faiss").exists()

if cache_exists and not FORCE_REEMBED:
    print(f"Loading cached FAISS index from {CACHE_DIR} ...")
    vectorstore = FAISS.load_local(
        str(CACHE_DIR), embeddings, index_name=INDEX_NAME, allow_dangerous_deserialization=True
    )
else:
    print(f"Embedding {len(row_docs)} drug rows with {config['embedding']['model_name']} ...")
    vectorstore = FAISS.from_documents(row_docs, embeddings)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    vectorstore.save_local(str(CACHE_DIR), index_name=INDEX_NAME)
    print(f"Cached embeddings to {CACHE_DIR}")

# Assert the COUNT, not just that something loaded — a partially-written or stale
# cache is a real failure mode and it is silent until retrieval quietly misses rows.
assert vectorstore.index.ntotal == len(row_docs), (
    f"Vector store has {vectorstore.index.ntotal} vectors for {len(row_docs)} rows — "
    "cache is stale or embedding collapsed. Set FORCE_REEMBED = True to rebuild."
)
assert vectorstore.index.d == config["embedding"]["dimension"], (
    f"Cached vectors are {vectorstore.index.d}-dim but config says "
    f"{config['embedding']['dimension']} — re-embed with FORCE_REEMBED = True."
)
print(f"✅ knowledge base: {vectorstore.index.ntotal} rows × {vectorstore.index.d} dims, via {config['embedding']['model_name']}")

/tmp/ipykernel_28323/3738461831.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
/home/bhargav/portfolio-project/healthins-enrollment-assistant-dev/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3951.50it/s]

Loading cached FAISS index from ../../data/processed/NC/2026/formulary/faiss_cache_sentence-transformers-all-MiniLM-L6-v2_384d ...
✅ knowledge base: 2394 rows × 384 dims, via sentence-transformers/all-MiniLM-L6-v2


## 3.5 Hybrid Retrieval (Dense + BM25)

`05-Hybrid_Search_Formulary.ipynb` tested combining this vector store with a lexical BM25 retriever (`langchain_community.retrievers.BM25Retriever`, fused via `EnsembleRetriever`) against the eval set in `04-Evaluate_Formulary_RAG.ipynb`, and found a real, measured improvement over vector-only search: Hit Rate at `top_k=3` went from 93.5% to 98.0%, MRR from 0.854 to 0.923 — with no added cloud dependency or cost, since BM25 is plain statistical term matching (no model, no LLM call).

That result is adopted here: retrieval below is hybrid (dense + BM25), not vector-only. A drug name + strength is an exact lexical string, and BM25 catches cases where the embedding model ranks a semantically-similar-but-wrong drug above the exact match.

In [4]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# Built from the same rows as the FAISS index — statistical term matching, no
# embeddings, no LLM call, so this adds no cost or new external dependency.
bm25_retriever = BM25Retriever.from_documents(row_docs)


def build_hybrid_retriever(k, vector_weight=0.5, bm25_weight=0.5):
    vector_retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    bm25_retriever.k = k
    return EnsembleRetriever(
        retrievers=[vector_retriever, bm25_retriever],
        weights=[vector_weight, bm25_weight],
    )


print("✅ Hybrid retriever ready (dense + BM25)")

✅ Hybrid retriever ready (dense + BM25)


## 4. LLM + Retrieval

Same local Ollama model as `langchain_03_table_aware.ipynb`, with the context wrapped in explicit `<START_OF_CONTEXT>`/`<END_OF_CONTEXT>` boundary tags (the `02-Basic_RAG.ipynb` augmentation template) rather than an unmarked context block. Retrieval itself now goes through the **hybrid retriever** built in §3.5 instead of a plain `similarity_search_with_score` call — see the note above for why, and `05-Hybrid_Search_Formulary.ipynb` for the measured comparison.

One trade-off from switching to hybrid: `EnsembleRetriever` returns fused, reranked documents without a single comparable distance score (its internal Reciprocal Rank Fusion score isn't attached to the returned docs), unlike FAISS's raw L2 distance. So below we print **rank order** rather than a numeric score — rank order is what the fusion actually optimizes, and it's still there to eyeball.

`ask()` also appends a **Sources** section to the answer (drug, page, formulary section), de-duped from the retrieved rows' metadata. A tier/quantity-limit claim is only useful to a member if they can go verify it against the actual PDF page — surfacing that at answer time, not just in a separate debug printout, is the difference between a demo and something a member could actually trust.

In [5]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=config["llm"]["model"],
    temperature=config["llm"]["temperature"],
    num_predict=config["llm"]["max_tokens"],
)

SYSTEM_INSTRUCTION = (
    "You are a helpful assistant answering questions about health insurance plans. "
    "Use only the provided context to answer. If the context doesn't contain the answer, say you don't know."
)

# Course-standard augmentation template (02-Basic_RAG.ipynb): context inside explicit boundary tags.
PROMPT_TEMPLATE = (
    "Read the following informations that might contain the context you require to "
    "answer the question. You can use the informations starting from the "
    "<START_OF_CONTEXT> tag and end with the <END_OF_CONTEXT> tag. Here is the content:\n\n"
    "<START_OF_CONTEXT>\n{context}\n<END_OF_CONTEXT>\n\n"
    "Please provide an informative and accurate answer to the following question based "
    "on the available context. Be concise and take your time.\nQuestion: {question}\nAnswer:"
)


def format_sources(docs):
    # De-dupe by drug name — a row can legitimately be retrieved more than once
    # across different questions, but never worth citing twice for one answer.
    seen = set()
    lines = []
    for doc in docs:
        drug = doc.metadata.get("drug", "unknown")
        if drug in seen:
            continue
        seen.add(drug)
        page = doc.metadata.get("page", "?")
        section = doc.metadata.get("section") or "N/A"
        lines.append(f"- {drug} — p.{page}, {section}")
    return "\n".join(lines)


def ask(question, k=None):
    k = k or config["retrieval"]["top_k"]
    retriever = build_hybrid_retriever(k)
    docs = retriever.invoke(question)[:k]
    context = "\n\n".join(d.page_content for d in docs)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    response = llm.invoke([
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": prompt},
    ])
    # Append the source rows the answer was grounded in — a formulary tier/limit claim
    # is only useful to a member if they can go verify it against the actual PDF page.
    answer = f"{response.content}\n\nSources:\n{format_sources(docs)}"
    return answer, docs


print(f"✅ LLM ready: {config['llm']['model']}")

✅ LLM ready: llama3.1


In [6]:
question = (
    "What is the tier and quantity limit for acetaminophen with codeine "
    "solution 120-12 mg/5ml and tablet 300-15 mg?"
)

answer, docs = ask(question)

print(f"=== Retrieved (top_k={config['retrieval']['top_k']}, hybrid dense+BM25 — order shown is the fused rank) ===")
for rank, doc in enumerate(docs, start=1):
    print(f"┌─ Rank {rank}")
    print("│  " + doc.page_content.replace("\n", " "))
    print("└" + "─" * 60)

print("\nAnswer:", answer)

=== Retrieved (top_k=3, hybrid dense+BM25 — order shown is the fused rank) ===
┌─ Rank 1
│  Section: OPIOID ANALGESICS Drug: acetaminophen w/ codeine soln 120-12 mg/5ml Tier: Tier 1 Requirements/Limits: QL (2700 ml every 30 days); Subject to initial 5- day limit for 19 and younger; 7-day initial limit for all other ages
└────────────────────────────────────────────────────────────
┌─ Rank 2
│  Section: OPIOID ANALGESICS Drug: acetaminophen w/ codeine tab 300-15 mg Tier: Tier 1 Requirements/Limits: QL (390 tabs every 30 days); Subject to initial 5- day limit for 19 and younger; 7-day initial limit for all other ages
└────────────────────────────────────────────────────────────
┌─ Rank 3
│  Section: OPIOID ANALGESICS Drug: acetaminophen w/ codeine tab 300-60 mg Tier: Tier 1 Requirements/Limits: QL (180 tabs every 30 days); Subject to initial 5- day limit for 19 and younger; 7-day initial limit for all other ages
└────────────────────────────────────────────────────────────

Answer: Based

## 5. Why Grounding Matters: With vs. Without Retrieval

**Backported from `02-Basic_RAG.ipynb` §10.** One passing retrieval-augmented answer isn't proof retrieval is doing anything — the model might already "know" a plausible-sounding tier. Ask the *same* question with and without the retrieved context, on a fact only this specific formulary PDF (not general training data) can supply: an exact quantity limit.

In [7]:
grounding_question = "What tier and requirements/limits apply to trandolapril tabs 1mg, 2mg, 4mg?"

print("WITHOUT CONTEXT:\n")
no_context_answer = llm.invoke([
    {"role": "system", "content": SYSTEM_INSTRUCTION},
    {"role": "user", "content": f"Be concise.\nQuestion: {grounding_question}\nAnswer:"},
]).content
print(no_context_answer)

WITHOUT CONTEXT:



I don't know.


In [8]:
print("WITH CONTEXT:\n")
with_context_answer, grounded_docs = ask(grounding_question)
print("Retrieved:", grounded_docs[0].page_content.replace("\n", " "))

print("\nAnswer:", with_context_answer)

WITH CONTEXT:



Retrieved: Section: ACE INHIBITORS Drug: trandolapril tabs 1mg, 2mg, 4mg Tier: Tier 1 Requirements/Limits: None

Answer: According to the context, the tier and requirements/limits for trandolapril tabs 1mg, 2mg, 4mg are:

Tier: Tier 1
Requirements/Limits: None

Sources:
- trandolapril tabs 1mg, 2mg, 4mg — p.37, ACE INHIBITORS
- tranexamic acid soln 1000mg/10ml; tabs 650mg — p.79, MISCELLANEOUS
- risperidone soln 1mg/ml; tabs .25mg, .5mg, 1mg, 2mg, 3mg, 4mg; tbdp .25mg, .5mg, 1mg, 2mg, 3mg, 4mg — p.54, ANTIPSYCHOTICS


**What just happened?** Without context, the model has no way to know this plan's actual tier/requirements for trandolapril — it either guesses at a plausible-sounding answer or, if instructed carefully, admits it doesn't know. With context, the retrieved row (`Section: ACE INHIBITORS, Tier: Tier 1, Requirements/Limits: None`) grounds the answer in the plan's real data. This is the same argument `02-Basic_RAG.ipynb` makes with its internal-release-notes demo, now proven on this project's own PDF: retrieval is what turns a plausible answer into a *correct* one.

## 🔑 Key Takeaways

- The chunk → embed → search → answer skeleton is provider- and format-agnostic — it works the same whether the source is a CSV of blog posts or a PDF's drug tables.
- **The chunker has to match the data.** Fixed-size character windows (right for prose) would have re-broken the table structure this project already fixed with row-based extraction — porting a technique means porting the *parts that still apply*, not the whole notebook verbatim.
- **A cache belongs to exactly one embedding model.** Keying the FAISS cache directory by model name + dimension means a config change points at a different cache, never a corrupted one — and skips re-embedding 2,394 rows when nothing changed.
- **Assert vector counts, not just that something loaded.** A stale or partial cache is a silent failure otherwise.
- FAISS's default index here is L2 distance — *lower is better* — the opposite direction from cosine similarity in `02-Basic_RAG.ipynb`. Know which metric you're reading before trusting a number.
- Grounding proof (with vs. without context, same question) is cheap to add and is the most convincing evidence retrieval is actually doing something — worth keeping in any RAG notebook, not just the course one.
- **Citations belong in the answer, not just the debug printout.** `ask()` now returns the drug/page/section each claim was grounded in alongside the LLM's response — for a formulary, "trust me" isn't good enough; a member needs to be able to go check the actual plan document.
- **Retrieval is now hybrid (dense + BM25), not vector-only — because the numbers said so.** `05-Hybrid_Search_Formulary.ipynb` measured a real improvement on this exact data (93.5%→98.0% hit rate at `top_k=3`) for zero added cost. That evidence, not intuition, is why `ask()` changed — the same discipline that made this project skip reranking and embedding fine-tuning for lack of a comparable payoff.